# Week 5, Notebook 2: Transformers with PyTorch
## Build, Train, and Use Transformer Models

**What you'll build:** A Transformer-based sequence classifier and explore pre-trained models with Hugging Face.

**New concepts:**
- `nn.TransformerEncoder` in PyTorch
- Causal masking for autoregressive models
- Transfer learning: using pre-trained Transformers
- Tokenization and the Hugging Face ecosystem

**Time estimate:** 60 minutes

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import math

torch.manual_seed(42)
np.random.seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

## Part 1: Transformer Encoder for Sequence Classification

In [ ]:
# ============================================================
# Transformer encoder for classifying sequences
# Task: classify sequences of numbers as "ascending" or "not"
# ============================================================

def generate_sequence_data(n_samples=2000, seq_len=8, vocab_size=50):
    """Generate sequences and labels: 1=ascending, 0=random."""
    X = []
    y = []
    for _ in range(n_samples):
        if np.random.rand() > 0.5:
            # Ascending sequence (with noise)
            start = np.random.randint(0, vocab_size - seq_len - 5)
            seq = np.sort(np.random.choice(range(start, start + seq_len + 5),
                                           size=seq_len, replace=False))
            X.append(seq)
            y.append(1)
        else:
            # Random sequence
            seq = np.random.randint(0, vocab_size, size=seq_len)
            X.append(seq)
            y.append(0)

    X = torch.LongTensor(np.array(X))
    y = torch.LongTensor(y)
    return X, y


X_data, y_data = generate_sequence_data(3000)
n_train = 2400
X_train, y_train = X_data[:n_train], y_data[:n_train]
X_val, y_val = X_data[n_train:], y_data[n_train:]

print(f"Training: {X_train.shape}, Validation: {X_val.shape}")
print(f"Sample ascending: {X_data[y_data == 1][0].numpy()}")
print(f"Sample random:    {X_data[y_data == 0][0].numpy()}")

In [ ]:
# ============================================================
# Transformer Classifier
# ============================================================

class PositionalEncoding(nn.Module):
    """Sinusoidal positional encoding."""
    def __init__(self, d_model, max_len=512):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))  # (1, max_len, d_model)

    def forward(self, x):
        return x + self.pe[:, :x.size(1), :]


class TransformerClassifier(nn.Module):
    """Transformer encoder for sequence classification."""

    def __init__(self, vocab_size, d_model=64, n_heads=4, n_layers=2,
                 d_ff=128, n_classes=2, max_len=100, dropout=0.1):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_encoder = PositionalEncoding(d_model, max_len)
        self.dropout = nn.Dropout(dropout)

        # Transformer encoder layers
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=d_ff,
            dropout=dropout, batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)

        # Classification head
        self.classifier = nn.Sequential(
            nn.Linear(d_model, d_model // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(d_model // 2, n_classes)
        )

    def forward(self, x):
        # Embed tokens + add position
        h = self.embedding(x) * math.sqrt(64)
        h = self.pos_encoder(h)
        h = self.dropout(h)

        # Transformer encoder
        h = self.transformer(h)

        # Pool: mean of all token representations
        h = h.mean(dim=1)  # (batch, d_model)

        # Classify
        return self.classifier(h)


model = TransformerClassifier(vocab_size=50, d_model=64, n_heads=4,
                               n_layers=2, d_ff=128, n_classes=2)
print(model)
print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# ============================================================
# Training loop
# ============================================================
optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
batch_size = 64

for epoch in range(60):
    model.train()
    epoch_loss = 0
    correct = 0
    total = 0

    # Mini-batch training
    perm = torch.randperm(n_train)
    for i in range(0, n_train, batch_size):
        idx = perm[i:i+batch_size]
        xb, yb = X_train[idx], y_train[idx]

        logits = model(xb)
        loss = criterion(logits, yb)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item() * len(xb)
        correct += (logits.argmax(1) == yb).sum().item()
        total += len(xb)

    # Validation
    model.eval()
    with torch.no_grad():
        val_logits = model(X_val)
        val_loss = criterion(val_logits, y_val).item()
        val_acc = (val_logits.argmax(1) == y_val).float().mean().item()
        train_acc = correct / total

    history['train_loss'].append(epoch_loss / total)
    history['val_loss'].append(val_loss)
    history['train_acc'].append(train_acc)
    history['val_acc'].append(val_acc)

    if epoch % 10 == 0:
        print(f"  Epoch {epoch:3d} | Train Acc: {train_acc:.3f} | Val Acc: {val_acc:.3f}")

print(f"\nFinal Val Accuracy: {history['val_acc'][-1]:.1%}")

In [ ]:
# Visualize training
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].plot(history['train_loss'], label='Train', alpha=0.8)
axes[0].plot(history['val_loss'], label='Val', alpha=0.8)
axes[0].set_title('Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.2)

axes[1].plot(history['train_acc'], label='Train', alpha=0.8)
axes[1].plot(history['val_acc'], label='Val', alpha=0.8)
axes[1].set_title('Accuracy')
axes[1].set_ylim(0.4, 1.05)
axes[1].legend()
axes[1].grid(True, alpha=0.2)

plt.suptitle('TRANSFORMER CLASSIFIER: Sequence Classification', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('w5_02_training.png', dpi=100, bbox_inches='tight')
plt.show()

## Part 2: Attention Visualization — What Does the Transformer See?

In [ ]:
# ============================================================
# Extract and visualize attention weights
# ============================================================
# We need to hook into the transformer to get attention

model.eval()
sample_idx = 0
x_sample = X_val[sample_idx:sample_idx+1]
y_true = y_val[sample_idx].item()

# Get prediction
with torch.no_grad():
    logit = model(x_sample)
    pred = logit.argmax(1).item()

print(f"Sequence:   {x_sample[0].numpy()}")
print(f"True label: {'ascending' if y_true else 'random'}")
print(f"Prediction: {'ascending' if pred else 'random'}")

# For visualization, compute attention manually
with torch.no_grad():
    h = model.embedding(x_sample) * math.sqrt(64)
    h = model.pos_encoder(h)

    # Get first layer's attention
    layer0 = model.transformer.layers[0]
    # Use the self_attn module directly
    attn_out, attn_weights = layer0.self_attn(h, h, h, need_weights=True)

attn_np = attn_weights[0].numpy()  # (seq_len, seq_len)
tokens = [str(t) for t in x_sample[0].numpy()]

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(attn_np, cmap='Blues')
ax.set_xticks(range(len(tokens)))
ax.set_yticks(range(len(tokens)))
ax.set_xticklabels(tokens, fontsize=10)
ax.set_yticklabels(tokens, fontsize=10)
ax.set_xlabel('Keys')
ax.set_ylabel('Queries')
label = 'ascending' if y_true else 'random'
ax.set_title(f'Attention Weights (Layer 1) — "{label}" sequence')

for i in range(len(tokens)):
    for j in range(len(tokens)):
        ax.text(j, i, f'{attn_np[i,j]:.2f}', ha='center', va='center',
               fontsize=8, color='white' if attn_np[i,j] > 0.2 else 'black')

plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.savefig('w5_02_attention_vis.png', dpi=100, bbox_inches='tight')
plt.show()

## Part 3: Causal (Autoregressive) Masking — How GPT Works

GPT-style models can only attend to **past** tokens (not future ones).
This is enforced with a **causal mask**: an upper-triangular matrix of -infinity.

In [ ]:
# ============================================================
# Causal masking demonstration
# ============================================================

seq_len = 6
tokens_demo = ["I", "love", "deep", "learn", "-ing", "!"]

# Causal mask: position i can only attend to positions <= i
causal_mask = torch.triu(torch.ones(seq_len, seq_len) * float('-inf'), diagonal=1)

print("Causal mask (GPT-style):")
print("0 = can attend, -inf = blocked")
for i in range(seq_len):
    row = []
    for j in range(seq_len):
        row.append("  0  " if causal_mask[i,j] == 0 else " -inf")
    print(f"  {tokens_demo[i]:6s}: {''.join(row)}")

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Bidirectional (BERT-style)
bidir = np.ones((seq_len, seq_len))
axes[0].imshow(bidir, cmap='Blues', vmin=0, vmax=1)
axes[0].set_title('Bidirectional Attention (BERT)')
axes[0].set_xticks(range(seq_len))
axes[0].set_yticks(range(seq_len))
axes[0].set_xticklabels(tokens_demo, fontsize=9)
axes[0].set_yticklabels(tokens_demo, fontsize=9)

# Causal (GPT-style)
causal_vis = np.tril(np.ones((seq_len, seq_len)))
axes[1].imshow(causal_vis, cmap='Blues', vmin=0, vmax=1)
axes[1].set_title('Causal Attention (GPT)')
axes[1].set_xticks(range(seq_len))
axes[1].set_yticks(range(seq_len))
axes[1].set_xticklabels(tokens_demo, fontsize=9)
axes[1].set_yticklabels(tokens_demo, fontsize=9)

plt.suptitle('BERT vs GPT: Attention Masks', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('w5_02_causal.png', dpi=100, bbox_inches='tight')
plt.show()

print("\nBERT: every token sees ALL other tokens (bidirectional)")
print("GPT:  each token only sees PAST tokens (causal/autoregressive)")
print("This is the key architectural difference between BERT and GPT.")

## Part 4: The Transformer Landscape — What's Out There

| Model | Type | Attention | Key Use Case |
|-------|------|-----------|-------------|
| **BERT** | Encoder | Bidirectional | Understanding (classification, NER) |
| **GPT** | Decoder | Causal | Generation (text, code) |
| **T5** | Encoder-Decoder | Both | Any text-to-text task |
| **ViT** | Encoder | Bidirectional | Image classification |
| **CLIP** | Dual Encoder | Cross-modal | Image-text matching |
| **Whisper** | Encoder-Decoder | Both | Speech recognition |

The Transformer architecture is now the foundation of virtually all state-of-the-art models in NLP, vision, speech, and multimodal AI.

## ✅ Week 5 Complete — Self-Assessment

### You should now be able to:
- [ ] Build a Transformer encoder from scratch (pure Python AND PyTorch)
- [ ] Explain self-attention: "each token attends to all others via Q, K, V"
- [ ] Explain multi-head attention: "parallel attention for different relationships"
- [ ] Explain positional encoding: "sinusoidal signal that breaks permutation invariance"
- [ ] Distinguish BERT (bidirectional) from GPT (causal)
- [ ] Train a Transformer classifier in PyTorch
- [ ] Visualize attention weights to understand what the model learns

## 🏆 Congratulations — You Completed All 5 Weeks!

### Your Learning Journey:
- **Week 1:** Neural networks from scratch — forward pass, backprop, ReLU
- **Week 2:** Deep networks in PyTorch — init, normalization, loss landscapes
- **Week 3:** Generative AI — VAEs, latent spaces, deployment
- **Week 4:** Graph Neural Networks — message passing, GCN, GraphSAGE
- **Week 5:** Transformers — attention, multi-head, positional encoding, GPT vs BERT

### What's Next?
1. **Fine-tune a pre-trained model** — Hugging Face makes this 10 lines of code
2. **Diffusion Models** — the architecture behind Stable Diffusion, DALL-E
3. **Reinforcement Learning from Human Feedback (RLHF)** — how ChatGPT is trained
4. **Build something real** — take a problem you care about and apply what you've learned

*"You've built every major architecture from scratch. Nothing in deep learning is magic anymore."*